In [2]:
!pip install pandas cassandra-driver numpy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 56.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.5/3.5 MB 46.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.7/16.7 MB 52.2 MB/s  0:00:006m0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7/7 [cassandra-driver][cassandra-driver]

[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip


In [3]:
import pandas as pd
import cassandra
import re
import os
import glob
import numpy as np
import json
import csv

In [7]:
import csv
import glob
import os

# 1. البحث التلقائي عن الملفات في أي مكان داخل الـ Codespace الحالي
# سيبحث في المجلد الحالي وكل المجلدات الفرعية عن ملفات الكاساندرا
file_pattern = "**/2018-*-events.csv"
file_path_list = sorted(glob.glob(file_pattern, recursive=True))

if not file_path_list:
    # محاولة إضافية للبحث من المجلد الرئيسي الفعلي للـ Codespace
    file_path_list = sorted(
        glob.glob(
            "/workspaces/Database-Engineering/**/2018-*-events.csv",
            recursive=True,
        )
    )

full_data_rows_list = []

print(f"🔄 تم العثور تلقائياً على {len(file_path_list)} ملفاً للدمج.")

if len(file_path_list) == 0:
    print(
        "❌ تحذير: لم يتم العثور على الملفات. تأكد من رفعها بشكل صحيح في القائمة الجانبية."
    )
else:
    # 2. قراءة ودمج كل الملفات
    for f in file_path_list:
        with open(f, "r", encoding="utf8", newline="") as csvfile:
            csvreader = csv.reader(csvfile)
            next(csvreader)  # تخطي سطر العناوين (Header) لكل ملف

            for line in csvreader:
                full_data_rows_list.append(line)

    print(f"📊 إجمالي السطور المستخرجة: {len(full_data_rows_list)}")

    # 3. تسجيل الـ Dialect وحفظ الملف الجديد بجانب الـ Notebook مباشرة
    csv.register_dialect(
        "myDialect", quoting=csv.QUOTE_ALL, skipinitialspace=True
    )
    output_file = "./event_datafile_new.csv"

    with open(output_file, "w", encoding="utf8", newline="") as f:
        writer = csv.writer(f, dialect="myDialect")

        # كتابة العناوين المطلوبة للمشروع
        writer.writerow(
            [
                "artist",
                "firstName",
                "gender",
                "itemInSession",
                "lastName",
                "length",
                "level",
                "location",
                "sessionId",
                "song",
                "userId",
            ]
        )

        # تصفية البيانات وإعادة كتابتها
        for row in full_data_rows_list:
            if row[0] == "":
                continue

            writer.writerow(
                (
                    row[0],  # artist
                    row[2],  # firstName
                    row[3],  # gender
                    row[4],  # itemInSession
                    row[5],  # lastName
                    row[6],  # length
                    row[7],  # level
                    row[8],  # location
                    row[12],  # sessionId
                    row[13],  # song
                    row[16],  # userId
                )
            )

    print(f"✨ نجاح! تم توليد الملف وتصفيته بنجاح في: {output_file}")

🔄 تم العثور تلقائياً على 30 ملفاً للدمج.
📊 إجمالي السطور المستخرجة: 8056
✨ نجاح! تم توليد الملف وتصفيته بنجاح في: ./event_datafile_new.csv


In [8]:
with open('event_datafile_new.csv', 'r', encoding = 'utf8') as f:
    print(sum(1 for line in f))

6821


In [10]:
from cassandra.cluster import Cluster
try:
    cluster = Cluster()

    # To establish connection and begin executing queries, need a session
    session = cluster.connect()
except Exception as e:
    print(e)

In [11]:
try:
    session.execute("""
    CREATE KEYSPACE IF NOT EXISTS musicdb
    WITH REPLICATION = 
    { 'class' : 'SimpleStrategy', 'replication_factor' : 1}""")
    
except Exception as e:
    print(e)

In [12]:
# Set KEYSPACE to the keyspace specified above
try:
    session.set_keyspace('musicdb')
except Exception as e:
    print(e)

In [13]:
query = """
        CREATE TABLE IF NOT EXISTS song_sessions (
            session_id int,
            item_in_session int,
            artist text,
            song text,
            length float,
            PRIMARY KEY (session_id, item_in_session)
        )
        """
try:
    session.execute(query)
except Exception as e:
    print(e)

In [14]:
file = 'event_datafile_new.csv'

with open(file, encoding = 'utf8') as f:
    csvreader = csv.reader(f)
    next(csvreader) # skip header
    for line in csvreader:
        query = "INSERT INTO song_sessions (session_id, item_in_session, artist, song, length)"
        query = query + "VALUES (%s, %s, %s, %s, %s)"
        ## Assign which column element should be assigned for each column in the INSERT statement.
        session.execute(query, (int(line[8]), int(line[3]), line[0], line[9], float(line[5])))

In [15]:
query = "SELECT artist, song, length FROM song_sessions WHERE session_id = 338 AND item_in_session = 4"

try:
    rows = session.execute(query)
    for row in rows:
        print(list(row))
except Exception as e:
    print(e)

['Faithless', 'Music Matters (Mark Knight Dub)', 495.30731201171875]


In [16]:
query = """
        CREATE TABLE IF NOT EXISTS user_song_sessions (
            user_id int,
            session_id int,
            item_in_session int,
            artist text,
            song text,
            first_name text,
            last_name text,
            PRIMARY KEY ((user_id, session_id), item_in_session)
        )
        """
try:
    session.execute(query)
except Exception as e:
    print(e)

In [17]:
file = 'event_datafile_new.csv'

with open(file, encoding = 'utf8') as f:
    csvreader = csv.reader(f)
    next(csvreader) # skip header
    for line in csvreader:
        query = "INSERT INTO user_song_sessions (user_id, session_id, item_in_session, artist, song, first_name, last_name)"
        query = query + "VALUES (%s, %s, %s, %s, %s, %s, %s)"
        ## Assign which column element should be assigned for each column in the INSERT statement.
        session.execute(query, (int(line[10]), int(line[8]), int(line[3]), line[0], line[9], line[1], line[4]))

In [18]:
query = "SELECT artist, song, first_name,last_name FROM user_song_sessions WHERE user_id = 10 AND session_id = 182"

try:
    rows = session.execute(query)
    for row in rows:
        print(list(row))
except Exception as e:
    print(e)

['Down To The Bone', "Keep On Keepin' On", 'Sylvie', 'Cruz']
['Three Drives', 'Greece 2000', 'Sylvie', 'Cruz']
['Sebastien Tellier', 'Kilometer', 'Sylvie', 'Cruz']
['Lonnie Gordon', 'Catch You Baby (Steve Pitron & Max Sanna Radio Edit)', 'Sylvie', 'Cruz']


In [19]:
query = """
        CREATE TABLE IF NOT EXISTS user_listen_songs (
            song text,
            user_id int,
            first_name text,
            last_name text,
            PRIMARY KEY (song, user_id)
        )
        """
try:
    session.execute(query)
except Exception as e:
    print(e)

In [20]:
file = 'event_datafile_new.csv'

with open(file, encoding = 'utf8') as f:
    csvreader = csv.reader(f)
    next(csvreader) # skip header
    for line in csvreader:
        query = "INSERT INTO user_listen_songs (song, user_id, first_name, last_name)"
        query = query + "VALUES (%s, %s, %s, %s)"
        ## Assign which column element should be assigned for each column in the INSERT statement.
        session.execute(query, (line[9], int(line[10]), line[1], line[4]))

In [21]:
query = "SELECT first_name,last_name FROM user_listen_songs WHERE song='All Hands Against His Own'"

try:
    rows = session.execute(query)
    for row in rows:
        print(list(row))
except Exception as e:
    print(e)

['Jacqueline', 'Lynch']
['Tegan', 'Levine']
['Sara', 'Johnson']
